In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
df = pd.read_csv("heart1.csv")

X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]


In [3]:
categorical_features = [
    "Sex",
    "ChestPainType",
    "RestingECG",
    "ExerciseAngina",
    "ST_Slope"
]

numerical_features = [
    "Age",
    "RestingBP",
    "Cholesterol",
    "FastingBS",
    "MaxHR",
    "Oldpeak"
]


In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [6]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42))
    ]
)

param_grid = {
    "classifier__n_estimators": [400, 600],
    "classifier__max_depth": [8, 10, 12],
    "classifier__min_samples_split": [5, 10],
    "classifier__min_samples_leaf": [1, 2],
    "classifier__max_features": ["sqrt"],
    "classifier__class_weight": ["balanced"]
}
grid_search = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid,
    cv=10,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_

print("Best RF parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)


Best RF parameters: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 10, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 400}
Best CV ROC-AUC: 0.9326390428677014


In [7]:
from sklearn.metrics import accuracy_score

y_pred = best_rf.predict(X_test)
print("Tes Accuracy:", accuracy_score(y_test, y_pred))


Tes Accuracy: 0.8913043478260869


In [8]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)# Predictions
y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]

print("TUNED RANDOM FOREST METRICS")
print("---------------------------")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


TUNED RANDOM FOREST METRICS
---------------------------
Accuracy : 0.8913043478260869
Precision: 0.8942307692307693
Recall   : 0.9117647058823529
F1-score : 0.9029126213592233
ROC-AUC  : 0.9314921090387375

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.87      0.88        82
           1       0.89      0.91      0.90       102

    accuracy                           0.89       184
   macro avg       0.89      0.89      0.89       184
weighted avg       0.89      0.89      0.89       184

Confusion Matrix:

[[71 11]
 [ 9 93]]


In [9]:
import pickle

pickle.dump(best_rf, open("heart_disease_pipeline.pkl", "wb"))


In [13]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Use your already-trained tuned pipeline
model = best_rf   # no pickle needed since notebook already has it

# ---- UI Widgets ----
age = widgets.IntText(description="Age:")
sex = widgets.Dropdown(
    options=["M", "F"],
    description="Sex:"
)

cp = widgets.Dropdown(
    options=["ATA", "NAP", "ASY", "TA"],
    description="ChestPain:"
)

bp = widgets.IntText(description="RestingBP:")
chol = widgets.IntText(description="Cholesterol:")
fbs = widgets.Dropdown(
    options=[0, 1],
    description="FastingBS:"
)

ecg = widgets.Dropdown(
    options=["Normal", "ST", "LVH"],
    description="RestingECG:"
)

maxhr = widgets.IntText(description="MaxHR:")

angina = widgets.Dropdown(
    options=["Y", "N"],
    description="ExerciseAngina:"
)

oldpeak = widgets.FloatText(description="Oldpeak:")

slope = widgets.Dropdown(
    options=["Up", "Flat", "Down"],
    description="ST_Slope:"
)

button = widgets.Button(description="Predict", button_style="success")
output = widgets.Output()

# ---- Prediction Logic ----
def predict(b):
    with output:
        clear_output()
        user_input = pd.DataFrame([{
            "Age": age.value,
            "Sex": sex.value,
            "ChestPainType": cp.value,
            "RestingBP": bp.value,
            "Cholesterol": chol.value,
            "FastingBS": fbs.value,
            "RestingECG": ecg.value,
            "MaxHR": maxhr.value,
            "ExerciseAngina": angina.value,
            "Oldpeak": oldpeak.value,
            "ST_Slope": slope.value
        }])

        prediction = model.predict(user_input)[0]

        if prediction == 1:
            print("⚠️ Heart Disease Detected")
        else:
            print("✅ No Heart Disease Detected")

button.on_click(predict)

display(
    age, sex, cp, bp, chol, fbs, ecg, maxhr, angina, oldpeak, slope,
    button, output
)


IntText(value=0, description='Age:')

Dropdown(description='Sex:', options=('M', 'F'), value='M')

Dropdown(description='ChestPain:', options=('ATA', 'NAP', 'ASY', 'TA'), value='ATA')

IntText(value=0, description='RestingBP:')

IntText(value=0, description='Cholesterol:')

...

Dropdown(description='ExerciseAngina:', options=('Y', 'N'), value='Y')

FloatText(value=0.0, description='Oldpeak:')

Dropdown(description='ST_Slope:', options=('Up', 'Flat', 'Down'), value='Up')

Button(button_style='success', description='Predict', style=ButtonStyle())

Output()